importing libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import geopandas as gpd

In [ ]:
first making a model based on just previous data without any sesmic data

In [ ]:
df = pd.read_csv("/content/query (4).csv")

print(df.columns)
print(df.head())

Index(['time', 'latitude', 'longitude', 'depth', 'mag', 'magType', 'nst',
       'gap', 'dmin', 'rms', 'net', 'id', 'updated', 'place', 'type',
       'horizontalError', 'depthError', 'magError', 'magNst', 'status',
       'locationSource', 'magSource'],
      dtype='object')
                       time  latitude  longitude  depth  mag magType    nst  \
0  2026-06-20T06:43:17.271Z   -7.0346   -13.2711   10.0  5.5      mb   89.0   
1  2026-06-20T03:49:06.278Z   52.7287   160.6412   35.0  5.8     mww   84.0   
2  2026-06-20T01:46:32.273Z  -57.1290   147.9179   10.0  5.7     mww   56.0   
3  2026-06-19T07:52:57.734Z   52.7355   161.1149   10.0  5.8     mww  102.0   
4  2026-06-19T06:52:31.597Z   52.7943   160.5652   10.0  6.6     mww  188.0   

    gap   dmin   rms  ...                   updated  \
0  60.0  1.401  0.62  ...  2026-06-21T07:12:15.583Z   
1  93.0  1.822  0.85  ...  2026-06-21T04:18:35.264Z   
2  73.0  6.748  1.93  ...  2026-06-21T02:09:51.495Z   
3  76.0  2.101  1.16  ...  2

In [ ]:
df["time"] = pd.to_datetime(df["time"])

print(df["time"].min())
print(df["time"].max())

2010-01-02 08:45:33.020000+00:00
2026-06-20 06:43:17.271000+00:00


In [ ]:
df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month

df[["time","year","month"]].head()

,time,year,month
0,2026-06-20 06:43:17.271000+00:00,2026,6
1,2026-06-20 03:49:06.278000+00:00,2026,6
2,2026-06-20 01:46:32.273000+00:00,2026,6
3,2026-06-19 07:52:57.734000+00:00,2026,6
4,2026-06-19 06:52:31.597000+00:00,2026,6


In [ ]:
df["lat_grid"] = (df["latitude"] // 5) * 5
df["lon_grid"] = (df["longitude"] // 5) * 5

In [ ]:
monthly = df.groupby(
    ["lat_grid","lon_grid","year","month"]
).agg(
    eq_count=("mag","count"),
    avg_mag=("mag","mean"),
    max_mag=("mag","max"),
    avg_depth=("depth","mean")
).reset_index()

In [ ]:
monthly["major_eq"] = (monthly["max_mag"] >= 6.5).astype(int)

monthly = monthly.sort_values(
    ["lat_grid", "lon_grid", "year", "month"]
)

monthly["target"] = (
    monthly.groupby(["lat_grid", "lon_grid"])["major_eq"]
    .shift(-1)
)



In [ ]:
monthly = monthly.dropna(subset=["target"])

print(monthly.shape)

(4858, 10)


In [ ]:
print(monthly["target"].value_counts())

target
0.0    4281
1.0     577
Name: count, dtype: int64


In [ ]:
features = [
    "eq_count",
    "avg_mag",
    "max_mag",
    "avg_depth"
]

X = monthly[features]
y = monthly["target"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.7992744860943168

Confusion Matrix:
[[651  77]
 [ 89  10]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.88      0.89      0.89       728
         1.0       0.11      0.10      0.11        99

    accuracy                           0.80       827
   macro avg       0.50      0.50      0.50       827
weighted avg       0.79      0.80      0.79       827



In [ ]:
the accuracy came to be 79.9 percent

In [ ]:
importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

print(
    importance.sort_values(
        by="Importance",
        ascending=False
    )
)

     Feature  Importance
3  avg_depth    0.708726
1    avg_mag    0.129299
2    max_mag    0.112676
0   eq_count    0.049299


now we check which features did the model used to predict

In [ ]:
monthly["prev_eq_count"] = (
    monthly.groupby(["lat_grid","lon_grid"])["eq_count"]
    .shift(1)
)

monthly["prev_max_mag"] = (
    monthly.groupby(["lat_grid","lon_grid"])["max_mag"]
    .shift(1)
)

In [ ]:
monthly["last3m_eq_count"] = (
    monthly.groupby(["lat_grid","lon_grid"])["eq_count"]
    .rolling(3)
    .sum()
    .reset_index(level=[0,1], drop=True)
)

In [ ]:
monthly = monthly.dropna()
print(monthly.shape)

(4132, 13)


In [ ]:
features = [
    "eq_count",
    "avg_mag",
    "max_mag",
    "avg_depth",
    "prev_eq_count",
    "prev_max_mag",
    "last3m_eq_count"
]

then we added some more features to help model pe=redict more accurately

In [ ]:
X = monthly[features]
y = monthly["target"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8355501813784765

Confusion Matrix:
[[683  45]
 [ 91   8]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.88      0.94      0.91       728
         1.0       0.15      0.08      0.11        99

    accuracy                           0.84       827
   macro avg       0.52      0.51      0.51       827
weighted avg       0.79      0.84      0.81       827



which gave us this 83.5 percent accuracy

now giving some sesmic data that is the active faults accross earth which is a great cause of earthquakes

In [ ]:
faults = gpd.read_file("/content/gem_active_faults.geojson")

In [ ]:
from shapely.geometry import Point

grid_points = gpd.GeoDataFrame(
    monthly,
    geometry=gpd.points_from_xy(
        monthly["lon_grid"],
        monthly["lat_grid"]
    ),
    crs="EPSG:4326"
)



In [ ]:
print(faults.geometry.geom_type.value_counts())


LineString    16195
Name: count, dtype: int64


In [ ]:
faults_proj = faults.to_crs("EPSG:3857")
grid_proj = grid_points.to_crs("EPSG:3857")

In [ ]:
from shapely.ops import nearest_points

def nearest_fault_distance(point, faults_gdf):
    return faults_gdf.distance(point).min()

grid_proj["distance_to_fault_m"] = grid_proj.geometry.apply(
    lambda x: nearest_fault_distance(x, faults_proj)
)

In [ ]:
grid_proj["distance_to_fault_km"] = (
    grid_proj["distance_to_fault_m"] / 1000
)

print(
    grid_proj["distance_to_fault_km"].describe()
)

count    4132.000000
mean      188.956436
std       193.522777
min         0.930756
25%        31.595322
50%       128.681999
75%       291.928951
max      1154.957401
Name: distance_to_fault_km, dtype: float64


In [ ]:
monthly["distance_to_fault_km"] = grid_proj["distance_to_fault_km"].values

In [ ]:
features = [
    "eq_count",
    "avg_mag",
    "max_mag",
    "avg_depth",
    "prev_eq_count",
    "prev_max_mag",
    "last3m_eq_count",
    "distance_to_fault_km"
]

In [ ]:
now adding the distance to nearest fault line data to model

In [ ]:
X = monthly[features]
y = monthly["target"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.8802902055622733
[[725   3]
 [ 96   3]]
              precision    recall  f1-score   support

         0.0       0.88      1.00      0.94       728
         1.0       0.50      0.03      0.06        99

    accuracy                           0.88       827
   macro avg       0.69      0.51      0.50       827
weighted avg       0.84      0.88      0.83       827



this finally gave us an accuracy of 88 percent with great accuracy towards earth quakes that is not harmful to us but our models tends to fail to perdict the hazardous earthquakes that is its limitation

An initial plate-boundary distance feature was engineered and evaluated. However, due to limitations in the available boundary dataset and its negative impact on model performance, it was excluded from the final model

SMOTE was applied to address class imbalance by oversampling the minority class. However, it did not significantly improve the model's ability to identify hazardous earthquakes. Therefore, the final model was trained without SMOTE